# 桥接到 Kubernetes：为什么需要 Device Plugin

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/lab-11-bridge-to-k8s.ipynb)

Colab 篇的收尾，也是全书主线的入口。用单机 GPU 的直觉，理解“Kubernetes 为什么看不到 GPU、Device Plugin 解决什么、HAMi 在哪里”，然后正式进入 K8s/HAMi 主线。

本 notebook 配合书中章节 [桥接到 Kubernetes：为什么需要 Device Plugin](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-11-bridge-to-k8s/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应书中一个 Step；Markdown 单元摘录了必要的解释，完整上下文请回到书中阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## Why this problem matters

回顾一下你在这十个 Lab 里做的事：打开一个 cell、`import torch`、用 GPU，甚至用 `nvcc` 编译了自己的 kernel。这一切之所以“能用”，是因为**你自己就是这台机器的管理员和唯一用户**：你知道有 GPU、知道怎么访问它、不需要跟别人协商。

但 Kubernetes 是个**多租户、多节点**的平台：成百上千个 Pod 在抢资源，调度器要决定“这个 Pod 去哪个节点、能不能拿到 GPU、能拿多少”。原生 Kubernetes 只认 CPU/内存这种可分割资源，**它根本不认识 GPU**。本 Lab 把这个“认知鸿沟”讲清楚，然后把你交接给 [foundations/](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/../foundations/) 和 [playground/](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/../playground/)。

## 单机 vs 平台：同一个 GPU，两套问题

| 你在 Colab 关心的 | Kubernetes 平台要解决的 |
| :--- | :--- |
| 我的 tensor 有没有上 GPU？ | 这个 Pod 该分到哪个节点？ |
| 显存够不够我的模型？ | 节点上 `nvidia.com/gpu` 还剩多少？ |
| 利用率高不高？ | 谁在共享这张卡、配额怎么给？ |
| 用完 `empty_cache` | 隔离：别人会不会抢我的显存？ |

## 为什么 Kubernetes 看不到 GPU

[Lab 02](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-02-gpu-on-pcie/) 里你看到 GPU 挂在 PCIe `00:04.0` 上。但 Kubernetes 的调度器只看 `node.status.allocatable`，默认那里面**只有 cpu/memory/ephemeral-storage**，没有 GPU。

![Device Plugin 把物理 GPU 上报为 node.allocatable 里的 nvidia.com/gpu，调度器据此匹配 Pod 请求，HAMi 再把一张卡切成多份并加配额](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-11-bridge-to-k8s/device-plugin-makes-gpu-visible.svg)

这就是 **Device Plugin** 存在的全部理由：在节点上发现 GPU，把它变成 `nvidia.com/gpu` 这个 Extended Resource 上报给 kubelet，让调度器“看见”。[Lab 06](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-06-nvml/) 你调的 NVML，就是 device plugin 用来发现设备的底层。

## HAMi 站在哪一层

原生 Device Plugin 只解决了“看得见”，但它只能整卡分配（`nvidia.com/gpu: 1` = 一整张）。而 [Lab 09](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-09-mps/)、[Lab 10](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-10-gpu-virtualization/) 告诉你：真实需求是**共享 + 配额**。HAMi 就补在 Device Plugin 之上：

- 把一张卡切成多个可调度份额；
- 给每个份额加 `gpumem`（[Lab 04](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-04-where-did-vram-go/) 的显存）和 `gpucores`（[Lab 05](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-05-gpu-utilization/) 的算力）配额；
- 让调度器感知这些细粒度资源。

你在 [L1 HAMi 第一份共享作业](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/../single-gpu/hami-first-share/) 会亲手验证这一切。

## 一个 Colab 能做的“桥接演示”

Colab 里没有 Kubernetes，但你可以模拟 Device Plugin 做的第一件事：**上报设备**，体会“把硬件翻译成资源”这一步：

In [ ]:
# 模拟 device plugin 对 kubelet 的上报：把 GPU 摘要成一条资源记录
!pip install -q nvidia-ml-py
from pynvml import *
nvmlInit()
h = nvmlDeviceGetHandleByIndex(0)
mem = nvmlDeviceGetMemoryInfo(h)
major, minor = nvmlDeviceGetCudaComputeCapability(h)
print("device report →")
print("  resource : nvidia.com/gpu")
print("  count    : 1")
print("  uuid     :", nvmlDeviceGetUUID(h))
print("  product  :", nvmlDeviceGetName(h))
print("  memory   :", round(mem.total/1024**2), "MiB")
print("  cc       :", f"{major}.{minor}")
print("  → kubelet 会把 count 写进 node.status.allocatable，GFD 会把 product/memory/cc 写成 node label")

这就是 [playground/local-fake-gpu](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/../playground/local-fake-gpu/) 里 fake-gpu-operator 在做的事，只不过它在一个 CPU 节点上**模拟**了这一步，让你没 GPU 也能验证控制面。

## 离开 Colab 之后

![离开 Colab 之后的学习路线：foundations、playground、single-gpu、serving、multi-node、standards](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-11-bridge-to-k8s/learning-path-after-colab.svg)

| 你要去的下一站 | 你会得到什么 | 需要的环境 |
| :--- | :--- | :--- |
| [foundations](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/../foundations/) | 控制面/数据平面、Device Plugin/DRA 的概念边界 | 仅阅读 |
| [playground](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/../playground/) | 无 GPU 也能验证调度证据链 | 笔记本 + Docker/kind |
| [single-gpu](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/../single-gpu/) | 真 GPU + HAMi 共享 | 一张 NVIDIA GPU |
| [serving](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/../serving/) | vLLM 推理 + 可观测 | 单 GPU |
| [multi-node](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/../multi-node/) | Kueue/Volcano 多租户 | 2–3 节点 |
| [standards](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/../standards/) | DRA/生产毕业/recipe | 变量较大 |

## Verification

完成后你应该能：

- 用一句话说清 Device Plugin 解决了什么（让 K8s 看见 GPU）；
- 解释为什么原生 K8s 默认不认识 GPU；
- 把 HAMi 定位在 Device Plugin 之上的哪一层；
- 知道接下来该去 foundations 还是 playground。

## Failure Modes（方向性）

| 困惑 | 去/读哪 |
| :--- | :--- |
| “我还想先理解控制面 vs 数据平面” | [foundations](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/../foundations/) |
| “我没 GPU，想先玩调度” | [playground（fake GPU）](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/../playground/) |
| “我有 GPU，想直接上 HAMi” | [single-gpu](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/../single-gpu/) |

## Summary Table：Colab 篇给你交付了什么

| 能力 | 来自 | K8s 主线的对应 |
| :--- | :--- | :--- |
| 读懂 nvidia-smi | Lab 01–02 | 节点 GPU 上报、DCGM |
| CUDA 可用性 | Lab 03 | 驱动/runtime/CDI，镜像里的 CUDA 版本约束 |
| 显存动态 | Lab 04 | HAMi `gpumem` |
| 利用率语义 | Lab 05 | HAMi `gpucores`、监控 |
| NVML 设备信息 | Lab 06 | device plugin 发现、GFD label |
| CUDA 编程模型、nvcc、Compute Capability | Lab 07 | 镜像编译目标、节点选择（`nvidia.com/cuda.*` label） |
| Kernel 并行模型、CUDA Runtime API | Lab 08 | HAMi-core 拦截的层 |
| 共享机制边界 | Lab 09–10 | HAMi 共享 vs MIG/MPS/vGPU |
| 桥接 K8s | 本 Lab | Device Plugin |

## 恭喜，你完成了 Colab 篇

到这里，你已经具备了进入全书 K8s/HAMi 主线所需的**全部 GPU 直觉**。继续阅读 [GPU 资源控制面与课程地图](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/../foundations/gpu-control-plane-map/)，正式开始 Kubernetes 之旅。

## 记录你的 Metadata Card

运行完成后，把实际数值回填到书中该章的 Metadata Card：运行日期、驱动 / CUDA 版本、关键输出数值、遇到的 Failure Mode。欢迎在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提交 PR 或 Issue。